In [1]:
import torch
import torchaudio
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import os
import random

In [2]:
epochs = 20
window_size = 400
hop_length = 160
number_of_mels = 64
model_path = f"models/model_{window_size}_{hop_length}_{number_of_mels}_{epochs}.pth"

In [3]:
class CNNLSTM(nn.Module):
    def __init__(self, n_mels=number_of_mels, num_classes=5, hidden_dim=128, num_layers=2):
        super(CNNLSTM, self).__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),

            nn.Conv2d(32, 64, kernel_size=(3,3), padding=(1,1)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )

        self.lstm = nn.LSTM(
            input_size=(n_mels//4)*64,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim*2, num_classes)

    def forward(self, x):
        # x: (batch, time, n_mels)
        x = x.unsqueeze(1)   # (batch, 1, time, n_mels)
        x = self.conv(x)     # (batch, c, time', mel')

        b, c, t, f = x.shape
        x = x.permute(0,2,1,3).contiguous().view(b, t, c*f)

        out, _ = self.lstm(x)

        # Instead of just last timestep, average across time
        out = out.mean(dim=1)

        out = self.dropout(out)
        out = self.fc(out)   # logits
        return out

In [4]:
csv_data = pd.read_csv('../dataset/SEP-28k_labels.csv')

In [5]:
def multi_label(row):
    return [
        float(row['Prolongation'] > 0),
        float(row['Block'] > 0),
        float((row['SoundRep'] > 0) or (row['WordRep'] > 0)),
        float(row['Interjection'] > 0),
        float(row['NoStutteredWords'] > 0),
    ]

clips_dir = '../dataset/clips/stuttering-clips/clips'
label_names = ['Prolongation', 'Block', 'Repetition', 'Interjection', 'NoStutteredWords']

csv_data['label'] = csv_data.apply(multi_label, axis=1)
csv_data['filepath'] = csv_data.apply(
    lambda x: os.path.join(clips_dir, f"{x['Show']}_{x['EpId']}_{x['ClipId']}.wav"), axis=1
)

In [6]:
len(csv_data)

28177

In [7]:
model = CNNLSTM()
model.load_state_dict(torch.load(model_path, map_location="cpu"))
model.eval()

CNNLSTM(
  (conv): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): MaxPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0, dilation=1, ceil_mode=False)
  )
  (lstm): LSTM(1024, 128, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=5, bias=True)
)

In [8]:
def preprocess_audio(filepath, sample_rate=16000, n_mels=number_of_mels):
    waveform, sr = torchaudio.load(filepath)

    if sr != sample_rate:
        waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)

    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)

    mel_spectrogram = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate,
        n_mels=n_mels,
        n_fft=window_size,
        hop_length=hop_length
    )(waveform)

    mel_spectrogram = (mel_spectrogram - mel_spectrogram.mean()) / (mel_spectrogram.std() + 1e-9)

    mel_spectrogram = mel_spectrogram.squeeze(0).T.unsqueeze(0)  # (1, time, n_mels)
    return mel_spectrogram

In [12]:
def take_random_audio_and_predict():
    row = csv_data.iloc[random.randint(0, len(csv_data))]
    audio_file = row['filepath']
    label = row['label']
    
    with torch.no_grad():
        audio_input = preprocess_audio(audio_file)  # path to your file
        outputs = model(audio_input)
    
        # Multi-label case (sigmoid)
        probs = torch.sigmoid(outputs).squeeze(0)
        predictions = (probs > 0.5).int()  # threshold at 0.5
    
    print(f'{audio_file = }')
    print(f'{label_names = }')
    print(f'{label = }')
    print("Raw output:", outputs)
    print("Probabilities:", probs)
    print("Predicted labels:", predictions)
    global count
    global counter_count
    if 1 in predictions:
        count += 1
    else:
        counter_count += 1

In [14]:
count = 0
counter_count = 0
for i in range(5):
    print("-------------------------------------------------")
    try:
        take_random_audio_and_predict()
    except RuntimeError as e:
        print("Skipping...")
    print("-------------------------------------------------")
print(f'{count = }')
print(f'{counter_count = }')

-------------------------------------------------
audio_file = '../dataset/clips/stuttering-clips/clips/WomenWhoStutter_57_191.wav'
label_names = ['Prolongation', 'Block', 'Repetition', 'Interjection', 'NoStutteredWords']
label = [0.0, 1.0, 1.0, 1.0, 0.0]
Raw output: tensor([[-5.3159, -4.9263, -3.9463, -6.8606, -4.4662]])
Probabilities: tensor([0.0049, 0.0072, 0.0190, 0.0010, 0.0114])
Predicted labels: tensor([0, 0, 0, 0, 0], dtype=torch.int32)
-------------------------------------------------
-------------------------------------------------
audio_file = '../dataset/clips/stuttering-clips/clips/WomenWhoStutter_32_23.wav'
label_names = ['Prolongation', 'Block', 'Repetition', 'Interjection', 'NoStutteredWords']
label = [0.0, 0.0, 0.0, 0.0, 1.0]
Raw output: tensor([[-5.2706, -4.9325, -3.9623, -6.8954, -4.4953]])
Probabilities: tensor([0.0051, 0.0072, 0.0187, 0.0010, 0.0110])
Predicted labels: tensor([0, 0, 0, 0, 0], dtype=torch.int32)
-------------------------------------------------
---